# FCS Project 2026
Kristian Perriu

Matteo La Paglia 958482 m.lapaglia@campus.unimib.it

In [1]:
import pandas as pd 
import numpy as np 

In [2]:
url = "https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=true"
storage_options = {"User-Agent": "Mozilla/5.0"}
gdp = pd.read_csv(url, storage_options=storage_options)
gdp

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
0,Afghanistan,AFG,2000,1617.8264,Asia
1,Afghanistan,AFG,2001,1454.1108,Asia
2,Afghanistan,AFG,2002,1774.3087,Asia
3,Afghanistan,AFG,2003,1815.9282,Asia
4,Afghanistan,AFG,2004,1776.9182,Asia
...,...,...,...,...,...
7440,Zimbabwe,ZWE,2021,4827.1094,Africa
7441,Zimbabwe,ZWE,2022,5036.7830,Africa
7442,Zimbabwe,ZWE,2023,5218.0454,Africa
7443,Zimbabwe,ZWE,2024,5211.8975,Africa


In [3]:
df = pd.read_csv("student_dropout_risk_2026.csv")
df

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,4,16.3,0,3,8,0,6,37,5.7,0
1,2,18,Other,USA,1,1,High,1,1,0,...,3,8.4,1,1,7,0,6,15,5.4,1
2,3,24,Other,USA,0,0,High,0,1,1,...,3,11.3,1,0,2,0,3,4,4.3,0
3,4,25,Female,USA,0,1,Middle,1,1,0,...,2,13.2,1,1,5,0,7,10,6.7,0
4,5,15,Female,Canada,0,0,Low,1,1,0,...,4,7.4,1,0,7,0,10,24,5.8,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,99996,25,Female,USA,0,0,Middle,1,1,1,...,10,6.2,0,0,4,0,6,11,5.9,0
99996,99997,24,Female,India,0,1,Middle,0,1,1,...,0,8.7,1,3,7,1,5,21,5.5,1
99997,99998,27,Female,USA,0,0,Middle,1,1,1,...,8,7.5,0,2,2,0,6,29,5.8,0
99998,99999,21,Male,UK,0,0,High,0,1,0,...,6,13.6,1,2,4,1,10,43,6.2,0


## Task 1

In order to create a dataframe containing all the students considered at risk, we filter the rows(unique students) and select the ones who satisfy all the conditions. We use the `&` operator to apply more than one condition at once. 

note: the number of the students at risk in the resulting table (197) seemed to be a lot lower than expected (their total number is 100k). So we considered the portion of students for each of the three filtered sets: 12.93%, 8.63%, 10.91%. If we consider those as probabilities and the events as indipendent, the resulting probability (to be at_risk) is ~0.1217 and 1000000 $\times$ 0.1217=121,7; that is a number in the same order of magnitude of the one we found. 
We also found out that across the full 100k students dataset, the baseline dropout rate is 32.0%, while inside the `at_risk` group, the dropout rate is up to 76.1%. Confirming that the applied conditions are consistent in isolating a small and high-risk group.

In [ ]:
at_risk = df[
    (df["study_mode"] == "Online")
    & (df["attendance_rate"] < 50)
    & (df["missed_deadlines"] > 5)
]
at_risk

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend
478,479,22,Male,Germany,0,1,High,0,0,1,...,8,8,1,3,30,6.1,1,0.545455,0.12,stable
581,582,17,Female,India,1,0,Low,1,1,1,...,11,6,0,8,0,7.1,1,0.750000,-0.57,declining
642,643,29,Male,Mexico,0,0,Low,1,1,1,...,9,10,0,7,9,4.0,1,0.666667,-0.20,stable
1216,1217,15,Male,UK,0,0,Low,1,1,1,...,7,7,0,5,11,5.4,1,0.777778,-1.25,declining
1316,1317,21,Male,Brazil,0,0,Low,0,1,1,...,17,5,0,4,6,8.1,1,0.666667,-0.23,stable


## Task 2

In order to create the `at_risk_by_course` we grouped by the `course_type` and used the agg method to count the students who dropped out and to calculate the dropout rate (considering that the dropout is boolean). Then we had to sort the values in a descending order to find that the highest dropout rate is observed in the Law & Social Sciences course type.


In [5]:
at_risk_by_course = (
    at_risk.groupby("course_type").agg(students=("dropout", "count"), dropout_rate=("dropout", "mean"))
    .sort_values(by="dropout_rate", ascending=False)
)

at_risk_by_course

,students,dropout_rate
course_type,,
Law & Social Sciences,29,0.896552
Business,47,0.765957
STEM,47,0.765957
Arts & Humanities,40,0.750000
Health Sciences,34,0.647059


## Task 3
After directly adding the `pass_rate` and `gpa_change` columns to the original dataframe, we initialized the new `gpa_trend` column to its baseline value (i.e. "stable") and used the `loc` method to locate and change that value to according to the intervals for `gpa_change`.

In [9]:
df["pass_rate"]=df["units_passed"]/df["units_enrolled"]
df["gpa_change"]=df["gpa_semester_2"]-df["gpa_semester_1"]

df["gpa_trend"] = "stable" #initiate all the values to "stable" as baseline 
df.loc[df["gpa_change"] > 0.5, "gpa_trend"] = "improving"
df.loc[df["gpa_change"] < -0.5, "gpa_trend"] = "declining"

df[["pass_rate","gpa_change","gpa_trend"]]

,pass_rate,gpa_change,gpa_trend
0,1.000000,-0.89,declining
1,0.785714,-0.03,stable
2,0.823529,-0.70,declining
3,0.846154,-0.41,stable
4,0.875000,-1.05,declining
...,...,...,...
99995,1.000000,-0.98,declining
99996,0.500000,0.21,stable
99997,1.000000,-0.57,declining
99998,0.714286,-0.99,declining


## Task 5

We can calculate the dropout rate as the mean of the column and to find the students with the biggest gpa drop we can introduce a new column which calculates the difference, sort the df according to that column (which nlargest does automatically but it can be done with a classic sort) and then we take the first 10 instances from that df.  

In [24]:
dropout_rate = df["dropout"].mean()
print(dropout_rate)

0.32004


In [25]:
df["gpa_drop"] = df["gpa_semester_1"] - df["gpa_semester_2"]
top_drops = df.nlargest(10, "gpa_drop")[["person_id", "gpa_drop", "gpa_semester_1", "gpa_semester_2"]]
top_drops

,person_id,gpa_drop,gpa_semester_1,gpa_semester_2
23225,23226,2.49,3.95,1.46
96224,96225,2.40,4.00,1.60
88489,88490,2.37,3.58,1.21
51824,51825,2.35,3.90,1.55
58935,58936,2.33,3.59,1.26
46714,46715,2.32,4.00,1.68
43035,43036,2.30,4.00,1.70
72663,72664,2.27,2.90,0.63
76705,76706,2.23,3.72,1.49
23427,23428,2.23,4.00,1.77


## Task 6
The task was solved by using slicing of the dataframe using the inclusive between condition and creating a new column to perserve the values. It could also be used by using pd.cut and also by a for loop but such solution is to be avoided as instructed. 

In [26]:
df.loc[df["weekly_work_hours"] == 0, "work_group"] = "0"
df.loc[df["weekly_work_hours"].between(1, 10), "work_group"] = "1-10"
df.loc[df["weekly_work_hours"].between(11, 20), "work_group"] = "11-20"
df.loc[df["weekly_work_hours"].between(21, 30), "work_group"] = "21-30"
df.loc[df["weekly_work_hours"] > 30, "work_group"] = "30"

## Task 7

To solve this exercise we used np.where and passed on a clause to filter out the data and the values of the corresponding score if the clause is to be true. The we used a group by and an the agg method to manipulate the rest of the columns after the group by 

In [27]:
tuition_score = np.where(df["tuition_up_to_date"] == 1, 2, 0)
stress_score = np.where(df["stress_level"] >= 8, 1, 0)
sleep_score = np.where(df["sleep_hours"] < 5, 1, 0)
work_score = np.where((df["part_time_job"] == 1) & (df["weekly_work_hours"] > 20), 1, 0)
attendence_score = np.where(df["attendance_rate"] < 60, 2, 0)
deadlines_score = np.where(df["missed_deadlines"] > 3, 1, 0)

df["risk_score"] = tuition_score + stress_score + sleep_score + work_score + attendence_score + deadlines_score
result = df.groupby(df["risk_score"]).agg(   
    n_students = ("risk_score", "size"),
    dropout_rate=("dropout", "mean")
)

result 


,n_students,dropout_rate
risk_score,,
0,7082,0.282265
1,6489,0.405301
2,31165,0.204268
3,26508,0.288441
4,15935,0.390901
5,8335,0.501620
6,3443,0.634911
7,927,0.721683
8,116,0.844828


## Task 8

Filtered the df by removing every country with OWID in the country code. 

In [28]:
gdp.code.unique()

<StringArray>
[     'AFG',      'ALB',      'DZA',      'AND',      'AGO',      'ATG',
      'ARG',      'ARM',      'ABW',      'AUS',
 ...
      'USA',      'VIR', 'OWID_UMC',      'URY',      'UZB',      'VUT',
      'VNM', 'OWID_WRL',      'ZMB',      'ZWE']
Length: 212, dtype: str

In [29]:
gdp_filtered = gdp[~gdp["code"].str.startswith("OWID")]
gdp_filtered

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
0,Afghanistan,AFG,2000,1617.8264,Asia
1,Afghanistan,AFG,2001,1454.1108,Asia
2,Afghanistan,AFG,2002,1774.3087,Asia
3,Afghanistan,AFG,2003,1815.9282,Asia
4,Afghanistan,AFG,2004,1776.9182,Asia
...,...,...,...,...,...
7440,Zimbabwe,ZWE,2021,4827.1094,Africa
7441,Zimbabwe,ZWE,2022,5036.7830,Africa
7442,Zimbabwe,ZWE,2023,5218.0454,Africa
7443,Zimbabwe,ZWE,2024,5211.8975,Africa


In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 37 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   person_id                     100000 non-null  int64  
 1   age                           100000 non-null  int64  
 2   gender                        100000 non-null  str    
 3   nationality                   100000 non-null  str    
 4   first_generation              100000 non-null  int64  
 5   disability_status             100000 non-null  int64  
 6   family_income_level           100000 non-null  str    
 7   scholarship_holder            100000 non-null  int64  
 8   tuition_up_to_date            100000 non-null  int64  
 9   part_time_job                 100000 non-null  int64  
 10  weekly_work_hours             100000 non-null  int64  
 11  course_type                   100000 non-null  str    
 12  study_mode                    100000 non-null  str    
 

## Task 9
First we need to prepare the dfs by filtering out all the irrelevant and duplicate data from the gdp, since only the enitty on owid_region are the only columns that matter for our task, we will remove the rest and remove every duplicate to avoid problematic merges, then we make the matching columns lowercase just for safety, we merge the dfs and impute the dropout rate for each country. 

In [35]:
# preparing dfs for merge 

countries = gdp_filtered[["entity", "owid_region"]].drop_duplicates().reset_index(drop=True)

countries["entity"] = countries["entity"].str.lower()
df["nationality"] = df["nationality"].str.lower()
countries

,entity,owid_region
0,afghanistan,Asia
1,albania,Europe
2,algeria,Africa
3,andorra,Europe
4,angola,Africa
...,...,...
200,uzbekistan,Asia
201,vanuatu,Oceania
202,vietnam,Asia
203,zambia,Africa


In [37]:
merged_df = pd.merge(df, countries, how="inner", left_on="nationality", right_on="entity")
merged_df.groupby("owid_region")["dropout"].mean().reset_index()

,owid_region,dropout
0,Africa,0.316721
1,Asia,0.322908
2,Europe,0.325639
3,North America,0.310621
4,Oceania,0.321981
5,South America,0.317620
